In [5]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import os
import cv2
from torchvision.transforms import ToTensor

dataset = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/GANS+PROD/Gan iter 1',
    annotation_csv='/home/demo-user/Downloads/all_annotations.csv'
)

image, target = dataset[0]
print(image.shape)           # Should be [3, H, W]
print(target['boxes'])       # Tensor of bounding boxes
print(target['labels'])      # Tensor([1])



class GalaxyDataset(Dataset):
    def __init__(self, image_dir, annotation_csv, transforms=None):
        self.image_dir = image_dir
        self.annotations = pd.read_csv(annotation_csv)
        self.transforms = transforms
        self.image_filenames = self.annotations["filename"].unique()

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        filename = self.image_filenames[idx]
        img_path = os.path.join(self.image_dir, filename)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = ToTensor()(image)  # Convert to PyTorch tensor [C,H,W]

        # Get all boxes for this image
        records = self.annotations[self.annotations["filename"] == filename]
        boxes = records[["xmin", "ymin", "xmax", "ymax"]].values
        boxes = torch.as_tensor(boxes, dtype=torch.float32)

        # All labels are "galaxy" → assume class ID = 1
        labels = torch.ones((boxes.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        return image, target


torch.Size([3, 512, 512])
tensor([[226., 227., 284., 285.]])
tensor([1])


In [7]:
dataset = GalaxyDataset(
    image_dir='/home/demo-user/Downloads/LoTSS files/GANS+PROD/Gan iter 1',
    annotation_csv='/home/demo-user/Downloads/all_annotations.csv'
)

image, target = dataset[0]
print(image.shape)           # Should be [3, H, W]
print(target['boxes'])       # Tensor of bounding boxes
print(target['labels'])      # Tensor([1])


torch.Size([3, 512, 512])
tensor([[226., 227., 284., 285.]])
tensor([1])
